# Naive Bayes Implementation

## Loading the labels

In [120]:
import os
import numpy as np
import pandas as pd
import re

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
labels_path = r'C:\Users\Kiel\Desktop\grad school\ai 201\label_folder\labels'
base = os.path.dirname(labels_path)
data_path = r'C:\Users\Kiel\Desktop\grad school\ai 201\data'

df = pd.read_csv(base + r'\labels', sep=' ', header=None, names=['label', 'path'])
df['path'] = df['path'].apply(lambda x: os.path.normpath(os.path.join(data_path, x)))
df['path'] = df['path']+'.eml'


# df.head()
# df.describe()
print(df.loc[0, 'path'])


C:\Users\Kiel\Desktop\grad school\ai 201\data\000\000.eml
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [116]:
exists = df['path'].apply(os.path.isfile)
print(exists.sum())
print(len(df)) # 40 files missing

df = df.loc[exists].reset_index(drop=True)
print(len(df))   # cleanup


37782
37822
37782


In [117]:
def split_data(df, train_ratio=0.7, seed=42):
    rng = np.random.default_rng(seed)
    train_idx, test_idx = [], []

    classes = df['label'].unique()
    for c in classes:
        c_idx = df.index[df['label'] == c].tolist()
        rng.shuffle(c_idx)
        split_point = int(len(c_idx) * train_ratio)
        train_idx.extend(c_idx[:split_point])
        test_idx.extend(c_idx[split_point:])

    return df.loc[train_idx].reset_index(drop=True), df.loc[test_idx].reset_index(drop=True)
    

train_df, test_df = split_data(df, train_ratio=0.7, seed=42)
train_df.to_csv("train_set.csv", index=False)
test_df.to_csv("test_set.csv", index=False)
print(train_df.label.value_counts())
print(test_df.label.value_counts())
print(len(train_df) / len(df))
print(len(test_df) / len(df))

label
spam    17417
ham      9030
Name: count, dtype: int64
label
spam    7465
ham     3870
Name: count, dtype: int64
0.6999894129479647
0.30001058705203537


## Tokenizer

In [ ]:
WORD_RE = re.compile(r'(?<=\s)[a-zA-Z]+(?=[\s,.])')

# (?<=\s) - whitespace in front
# [a-zA-Z]+ - alphabetic characters
# (?=[\s,.]) - whitespace, comma, or period at end

def read_email(path):
    with open(path, encoding='latin-1') as f:
        return f.read()

def tokenize(text):
    padded = ' ' + text + ' '
    return set(w.lower() for w in WORD_RE.findall(padded))

In [122]:
cases = {
    "Hello world, buy now. (free) click! the end": {'hello', 'world', 'buy', 'now', 'the', 'end'},
    "don't e-mail me":                              {'me'},
    "price $99 or 99cents":                         {'price', 'or'},
    "word.another":                                 set(),
    "tab\tsep\nnewline":                            {'tab', 'sep', 'newline'},
}
for text, expected in cases.items():
    got = tokenize(text)
    print('OK ' if got == expected else 'FAIL', repr(text), got)

OK  'Hello world, buy now. (free) click! the end' {'now', 'hello', 'world', 'buy', 'the', 'end'}
OK  "don't e-mail me" {'me'}
OK  'price $99 or 99cents' {'price', 'or'}
FAIL 'word.another' {'word'}
OK  'tab\tsep\nnewline' {'newline', 'sep', 'tab'}
